In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *

In [0]:
catalog           = "charles_schwab_retailbrokerage_dev_team_lemma"
staging_scd2      = f"{catalog}.staging.customer_scd2_versions"
silver_taxrate    = f"{catalog}.silver.taxrate"
staging_prospect  = f"{catalog}.staging.prospect_current"
gold_dim_customer = f"{catalog}.gold.dim_customer"

dbutils.widgets.text("batch_id", "1", "Batch ID")
current_batch = dbutils.widgets.get("batch_id")

# try:
#     carried_run_id = spark.sql(f"SELECT _run_id FROM {staging_scd2} LIMIT 1").first()[0]
# except:
#     carried_run_id = "unknown"

try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {staging_scd2} LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else "unknown"
except Exception:
    carried_run_id = "unknown"
    carried_batch = "unknown"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold;")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_customer_customer', 'Starting processing for gold dim_customer')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'RUNNING')

In [0]:
df_dim_customer = spark.sql(f"""
    WITH prospect_enrichment AS (
        SELECT 
            agency_id, TRY_CAST(credit_rating AS INT) as creditrating, TRY_CAST(net_worth AS DECIMAL(15,2)) as networth,
            CONCAT_WS('+',
                CASE WHEN TRY_CAST(net_worth AS DECIMAL(15,2)) > 1000000 OR TRY_CAST(annual_income AS DECIMAL(15,2)) > 200000 THEN 'HighValue' ELSE NULL END,
                CASE WHEN TRY_CAST(number_children AS INT) > 3 THEN 'Expenses' ELSE NULL END,
                CASE WHEN TRY_CAST(age AS INT) BETWEEN 45 AND 67 THEN 'Boomer' ELSE NULL END,
                CASE WHEN TRY_CAST(age AS INT) > 67 THEN 'Elderly' ELSE NULL END,
                CASE WHEN TRY_CAST(age AS INT) < 25 THEN 'YoungAdult' ELSE NULL END,
                CASE WHEN TRY_CAST(net_worth AS DECIMAL(15,2)) BETWEEN 100000 AND 1000000 THEN 'Millennial' ELSE NULL END,
                CASE WHEN TRY_CAST(number_credit_cards AS INT) > 5 THEN 'Spender' ELSE NULL END,
                CASE WHEN own_or_rent = 'R' THEN 'Renter' ELSE NULL END
            ) AS marketingnameplate,
            last_name, first_name, address_line1, address_line2, postal_code
        FROM {staging_prospect}
    )
    SELECT
        CAST(CONCAT(DATE_FORMAT(s.action_ts, 'yyyyMMdd'), CAST(s.customerid AS STRING)) AS BIGINT) AS sk_customerid,
        s.customerid, s.taxid, 'ACTIVE' AS status, s.lastname, s.firstname, s.middleinitial, 
        COALESCE(UPPER(s.gender), 'U') AS gender
        , s.tier, s.dob,
        s.addressline1, s.addressline2, s.postalcode, s.city, s.stateprov, s.country,
        CASE WHEN s.c_local_1 IS NOT NULL THEN CONCAT_WS('-', s.c_ctry_1, s.c_area_1, s.c_local_1) ELSE NULL END AS phone1,
        CASE WHEN s.c_local_2 IS NOT NULL THEN CONCAT_WS('-', s.c_ctry_2, s.c_area_2, s.c_local_2) ELSE NULL END AS phone2,
        CASE WHEN s.c_local_3 IS NOT NULL THEN CONCAT_WS('-', s.c_ctry_3, s.c_area_3, s.c_local_3) ELSE NULL END AS phone3,
        s.primaryemail, s.alternateemail,
        tn.TX_NAME AS nationaltaxratedesc, CAST(tn.TX_RATE AS DECIMAL(6,4)) AS nationaltaxrate,
        tl.TX_NAME AS localtaxratedesc, CAST(tl.TX_RATE AS DECIMAL(6,4)) AS localtaxrate,
        
        -- FIX: Use Window Functions to forward-fill prospect data across all SCD-2 versions for the customer
        MAX(p.agency_id) OVER (PARTITION BY s.customerid) AS agencyid, 
        MAX(p.creditrating) OVER (PARTITION BY s.customerid) AS creditrating, 
        MAX(p.networth) OVER (PARTITION BY s.customerid) AS networth, 
        MAX(p.marketingnameplate) OVER (PARTITION BY s.customerid) AS marketingnameplate,
        
        s.iscurrent, CAST(s.action_ts AS DATE) as valid_from, s.enddate as valid_to, 
        CAST(s.action_ts AS DATE) as effectivedate, s.enddate, s.version_number, s.record_hash,
        CURRENT_TIMESTAMP() AS system_valid_from, CAST('9999-12-31 23:59:59' AS TIMESTAMP) AS system_valid_to,
        s._batch, CURRENT_TIMESTAMP() AS _load_ts, '{carried_run_id}' AS _run_id
    FROM {staging_scd2} s
    LEFT JOIN {silver_taxrate} tn ON s.c_nat_tx_id = tn.TX_ID
    LEFT JOIN {silver_taxrate} tl ON s.c_lcl_tx_id = tl.TX_ID
    LEFT JOIN prospect_enrichment p 
        ON UPPER(s.lastname) = UPPER(p.last_name) AND UPPER(s.firstname) = UPPER(p.first_name)
       AND UPPER(s.addressline1) = UPPER(p.address_line1) AND UPPER(s.postalcode) = UPPER(p.postal_code)
""")

df_dim_customer.createOrReplaceTempView("v_dim_customer")

In [0]:

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_dim_customer} (
        sk_customerid BIGINT NOT NULL, customerid BIGINT, taxid STRING, status STRING, lastname STRING, 
        firstname STRING, middleinitial STRING, gender STRING, tier TINYINT, dob DATE, addressline1 STRING, 
        addressline2 STRING, postalcode STRING, city STRING, stateprov STRING, country STRING, phone1 STRING, 
        phone2 STRING, phone3 STRING, primaryemail STRING, alternateemail STRING, nationaltaxratedesc STRING, 
        nationaltaxrate DECIMAL(6,4), localtaxratedesc STRING, localtaxrate DECIMAL(6,4), agencyid STRING, 
        creditrating INT, networth DECIMAL(15,2), marketingnameplate STRING, iscurrent BOOLEAN, valid_from DATE, 
        valid_to DATE, effectivedate DATE, enddate DATE, version_number BIGINT, record_hash STRING, 
        system_valid_from TIMESTAMP, system_valid_to TIMESTAMP, _batch STRING, _load_ts TIMESTAMP, _run_id STRING
    ) USING DELTA
""")

spark.sql(f"""
    MERGE INTO {gold_dim_customer} target
    USING v_dim_customer source
    ON target.customerid = source.customerid AND target.version_number = source.version_number
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

dim_customer_count = spark.sql(f"SELECT COUNT(*) FROM {gold_dim_customer}").first()[0]
print(f"gold.dim_customer rows: {dim_customer_count} (Expected: 21,890)")

In [0]:
# ─── LOGGING ─────────────────────────────────────────────────────────────
staging_count = spark.sql(f"SELECT COUNT(*) FROM {staging_scd2}").first()[0]
log_audit_event(spark, carried_run_id, current_batch, "gold", "dim_customer", "MERGE", dim_customer_count)
log_pipeline_recon(spark, carried_run_id, current_batch, "CUSTOMER", "dim_customer", "staging", "gold", staging_count, dim_customer_count)

log_gold_recon(spark, carried_run_id, "gold.dim_customer", expected_count=21890, actual_count=dim_customer_count)
    
log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_customer_customer', 'Successfully completed processing for gold dim_customer.')